# Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.functions import col,trim, length

# Reading from Bronze Table

In [0]:
df = spark.table('workspace.bronze.crm_sales_details')

# Silver Transformation

## Trimming

In [0]:
for field in df.schema.fields:
  if isinstance(field.dataType, StringType):
    df = df.withColumn(field.name, trim(col(field.name)))

In [0]:
df.display()

# Cleaning Dates

## What the below code does.
We have 3 Columns.
1. sls_order_dt
2. sls_ship_dt
3. sls_due_dt

- The dates are in string format. We will be coneverting them to Spark Date_Format for Slicing, Filtering, Time-Series Analysis, etc.
- First, We will check if the data is null or if it is not 8 char -> none. Else, Convert it into Date format using to_date function by first converting it into string and then to dateformat.

In [0]:
df = (
    df
    .withColumn(
        "sls_order_dt",
        F.when(
            (col("sls_order_dt") == 0) | (length(col("sls_order_dt")) != 8),
            None
        ).otherwise(F.to_date(col("sls_order_dt").cast("string"), "yyyyMMdd"))
    )
    .withColumn(
        "sls_ship_dt",
        F.when(
            (col("sls_ship_dt") == 0) | (length(col("sls_ship_dt")) != 8),
            None
        ).otherwise(F.to_date(col("sls_ship_dt").cast("string"), "yyyyMMdd"))
    )
    .withColumn(
        "sls_due_dt",
        F.when(
            (col("sls_due_dt") == 0) | (length(col("sls_due_dt")) != 8),
            None
        ).otherwise(F.to_date(col("sls_due_dt").cast("string"), "yyyyMMdd"))
    )
)

# Sales and Price Prediction

The below code is for smart imputation strategy. 
- It check is sls_price is missing or invalid (Zero/negative). To handle Division by Zero erros.
- If sls_price is valid it keeps the original value.
- If the value is bad, it attempts to calculate the Price = Sales/Quantity.
- If the value is 0, it assigns None.

In [0]:
df = (
  df
  .withColumn(
    'sls_price',
    F.when(
      (col("sls_price").isNull()) | (col("sls_price") <= 0),
      F.when(
        col("sls_sales").isNotNull(),
        col("sls_sales") / col("sls_quantity")
      ).otherwise(None)
    ).otherwise(col("sls_price"))
  )
)

In [0]:
df.limit(10).display()

# Renaming Columns

In [0]:
RENAME_MAP = {
    "sls_ord_num": "order_number",
    "sls_prd_key": "product_number",
    "sls_cust_id": "customer_id",
    "sls_order_dt": "order_date",
    "sls_ship_dt": "ship_date",
    "sls_due_dt": "due_date",
    "sls_sales": "sales_amount",
    "sls_quantity": "quantity",
    "sls_price": "price"
}

for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name,new_name)

# Sanity checks for DataFrame

In [0]:
df.limit(10).display()

# Writing into Silver Table

In [0]:
df.write.mode('overwrite').format('delta').saveAsTable('workspace.silver.crm_sales')

In [0]:
%sql
Select * from workspace.silver.crm_sales limit 10
